# Libraries/Packages to Download

# Libraries to Import

In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import pickle
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from sklearn.preprocessing import LabelEncoder
import shap

C:\Users\cberches_acquiro\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Data Gathering

In [2]:
df = pd.read_csv('../datasets/WA_Fn-UseC_-HR-Employee-Attrition.csv')

# Machine Learning

In [3]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [4]:
education_map = {
    1: 'Below College',
    2: 'College',
    3: 'Bachelor',
    4: 'Master',
    5: 'Doctor'
}

environment_satisfaction_map = {
    1: 'Low',
    2: 'Medium',
    3: 'High',
    4: 'Very High'
}

job_involvement_map = {
    1: 'Low',
    2: 'Medium',
    3: 'High',
    4: 'Very High'
}

job_satisfaction_map = {
    1: 'Low',
    2: 'Medium',
    3: 'High',
    4: 'Very High'
}

performance_rating_map = {
    1: 'Low',
    2: 'Good',
    3: 'Excellent',
    4: 'Outstanding'
}

relationship_satisfaction_map = {
    1: 'Low',
    2: 'Medium',
    3: 'High',
    4: 'Very High'
}

work_life_balance_map = {
    1: 'Bad',
    2: 'Good',
    3: 'Better',
    4: 'Best'
}

In [5]:
#https://www.beresfordresearch.com/age-range-by-generation/
def generation(age):
    if (age >= 97) & (age <= 102):
        return 'WWII'
    elif (age >= 79) & (age <= 96):
        return 'Post War'
    elif (age >= 70) & (age <= 78):
        return 'Boomers I*'
    elif (age >= 60) & (age <= 69):
        return 'Boomers II (a/k/a Generation Jones)*'
    elif (age >= 44) & (age <= 59):
        return 'Gen X'
    elif (age >= 28 ) & (age <= 43):
        return 'Millennials'
    elif (age >= 12 ) & (age <= 27):
        return 'Gen Z'

In [6]:
def distance_category(distance):
    if (distance >= 1) & (distance <= 10):
        return 'Short Distance'
    elif (distance >= 11) & (distance <= 20):
        return 'Medium Distance'
    elif (distance >= 21) & (distance <= 30):
        return 'Long Distance'

In [7]:
def encode(list_to_encode, df):
    encoders = {}
    _ = df.copy()
    for col in list_to_encode:
        encoder = LabelEncoder()
        encoder.fit(_[col])
        encoders[col] = encoder
        _[col] = encoder.transform(_[col])
        label_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_)))
        print(label_mapping)
    return encoders, _

In [8]:
df['Education'] = df['Education'].map(education_map)
df['EnvironmentSatisfaction'] = df['EnvironmentSatisfaction'].map(environment_satisfaction_map)
df['JobInvolvement'] = df['JobInvolvement'].map(job_involvement_map)
df['JobSatisfaction'] = df['JobSatisfaction'].map(job_satisfaction_map)
df['PerformanceRating'] = df['PerformanceRating'].map(performance_rating_map)
df['RelationshipSatisfaction'] = df['RelationshipSatisfaction'].map(relationship_satisfaction_map)
df['WorkLifeBalance'] = df['WorkLifeBalance'].map(work_life_balance_map)
df['Overtime_Preference'] = df['OverTime'].map({'Yes': 1, 'No': 0})

In [9]:
df['AgeGeneration'] = df['Age'].apply(generation)
df['DistanceFromHomeCategory'] = df['DistanceFromHome'].apply(distance_category)

In [10]:
df = df.reindex(sorted(df.columns), axis=1)

In [11]:
churned_df = df.copy()[['AgeGeneration', 'Attrition', 'BusinessTravel', 'DailyRate',
       'Department', 'DistanceFromHome', 'DistanceFromHomeCategory',
       'Education', 'EducationField', 'EnvironmentSatisfaction',
        'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'NumCompaniesWorked',
    'OverTime', 'Overtime_Preference', 'PercentSalaryHike',
       'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']]

In [12]:
val_count = int(churned_df.shape[0] * .10)

In [13]:
toEncode_Orig = churned_df.dtypes[churned_df.dtypes == 'object'].index.tolist()

encoders, churned_df_encoded = encode(toEncode_Orig, churned_df)

{'Boomers II (a/k/a Generation Jones)*': 0, 'Gen X': 1, 'Gen Z': 2, 'Millennials': 3}
{'No': 0, 'Yes': 1}
{'Non-Travel': 0, 'Travel_Frequently': 1, 'Travel_Rarely': 2}
{'Human Resources': 0, 'Research & Development': 1, 'Sales': 2}
{'Long Distance': 0, 'Medium Distance': 1, 'Short Distance': 2}
{'Bachelor': 0, 'Below College': 1, 'College': 2, 'Doctor': 3, 'Master': 4}
{'Human Resources': 0, 'Life Sciences': 1, 'Marketing': 2, 'Medical': 3, 'Other': 4, 'Technical Degree': 5}
{'High': 0, 'Low': 1, 'Medium': 2, 'Very High': 3}
{'Female': 0, 'Male': 1}
{'High': 0, 'Low': 1, 'Medium': 2, 'Very High': 3}
{'Healthcare Representative': 0, 'Human Resources': 1, 'Laboratory Technician': 2, 'Manager': 3, 'Manufacturing Director': 4, 'Research Director': 5, 'Research Scientist': 6, 'Sales Executive': 7, 'Sales Representative': 8}
{'High': 0, 'Low': 1, 'Medium': 2, 'Very High': 3}
{'Divorced': 0, 'Married': 1, 'Single': 2}
{'No': 0, 'Yes': 1}
{'Excellent': 0, 'Outstanding': 1}
{'High': 0, 'Low': 1

In [14]:
df_encoded_val = churned_df_encoded.sample(n=val_count, random_state=42)
df_encoded_train = churned_df_encoded.drop(df_encoded_val.index)

In [15]:
features = ['AgeGeneration', 'BusinessTravel', 'DailyRate',
       'Department', 'DistanceFromHomeCategory',
       'Education', 'EducationField', 'EnvironmentSatisfaction',
       'Gender', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'NumCompaniesWorked',
        'Overtime_Preference', 'PercentSalaryHike',
       'PerformanceRating', 
       'RelationshipSatisfaction', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']
target = 'Attrition'

In [16]:
models = [LogisticRegression(random_state=42), RandomForestClassifier(random_state=42), GaussianNB(), DecisionTreeClassifier(random_state=42), GradientBoostingClassifier(random_state=42), AdaBoostClassifier(random_state=42)]

In [17]:
 X_train, X_test, y_train, y_test = train_test_split(df_encoded_train[features], df_encoded_train[target], test_size=0.2, random_state=42)

for index in range(len(models)):
    print('------------------------------------')
    print(type(models[index]))
    models[index].fit(X_train, y_train)
    
    y_pred = models[index].predict(X_test)
    y_pred_proba =  models[index].predict_proba(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print()
    print(f'Accuracy: {accuracy:.2f}')
    print(f'Precision: {precision:.2f}')
    print(f'Recall: {recall:.2f}')
    print(f'F1 Score: {f1:.2f}')
    
    cm = confusion_matrix(y_test, y_pred)
    print()
    
    print(cm)
    '''
    plt.clf()

    sns.set(font_scale=.8)
    plt.figure(figsize=(6,6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Active", "Churned"], yticklabels=["Active", "Churned"])
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix of Training Data')
    plt.show()'''
    
    if  index != 0 | index != 2 :
        importance_scores =  models[index].feature_importances_
    
        # Create a dataframe to display the feature importance scores
        importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importance_scores})
        importance_df = importance_df.sort_values('Importance', ascending=False)
        print()
        print(importance_df)

------------------------------------
<class 'sklearn.linear_model._logistic.LogisticRegression'>

Accuracy: 0.87
Precision: 0.60
Recall: 0.24
F1 Score: 0.35

[[222   6]
 [ 28   9]]
------------------------------------
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



Accuracy: 0.87
Precision: 0.80
Recall: 0.11
F1 Score: 0.19

[[227   1]
 [ 33   4]]
------------------------------------
<class 'sklearn.naive_bayes.GaussianNB'>

Accuracy: 0.87
Precision: 0.52
Recall: 0.59
F1 Score: 0.56

[[208  20]
 [ 15  22]]
------------------------------------
<class 'sklearn.tree._classes.DecisionTreeClassifier'>

Accuracy: 0.73
Precision: 0.13
Recall: 0.16
F1 Score: 0.14

[[188  40]
 [ 31   6]]
------------------------------------
<class 'sklearn.ensemble._gb.GradientBoostingClassifier'>

Accuracy: 0.86
Precision: 0.53
Recall: 0.27
F1 Score: 0.36

[[219   9]
 [ 27  10]]
------------------------------------
<class 'sklearn.ensemble._weight_boosting.AdaBoostClassifier'>

Accuracy: 0.87
Precision: 0.56
Recall: 0.38
F1 Score: 0.45

[[217  11]
 [ 23  14]]


The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


In [18]:
estimators_models = [('lr',LogisticRegression(random_state=42)), ('rfc', RandomForestClassifier(random_state=42)), 
              ('gnb', GaussianNB()), ('dtc', DecisionTreeClassifier(random_state=42)), ('gbc', GradientBoostingClassifier(random_state=42)),
                     ('abc', AdaBoostClassifier(random_state=42))]

In [19]:
model = VotingClassifier(
        estimators=estimators_models,
        voting='soft'
    )

print(type(model))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba =  model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print()
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

cm = confusion_matrix(y_test, y_pred)
print()

print(cm)


<class 'sklearn.ensemble._voting.VotingClassifier'>


lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.



Accuracy: 0.86
Precision: 0.50
Recall: 0.22
F1 Score: 0.30

[[220   8]
 [ 29   8]]


In [20]:
model = VotingClassifier(
        estimators=estimators_models,
        voting='hard'
    )

print(type(model))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print()
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

cm = confusion_matrix(y_test, y_pred)
print()

print(cm)


<class 'sklearn.ensemble._voting.VotingClassifier'>


lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



Accuracy: 0.87
Precision: 0.58
Recall: 0.19
F1 Score: 0.29

[[223   5]
 [ 30   7]]


The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


## Undersample

In [21]:
val_undersample = np.abs(churned_df_encoded[churned_df_encoded['Attrition'] == 1].shape[0] - churned_df_encoded[churned_df_encoded['Attrition'] == 0].shape[0])

In [22]:
rows_to_drop = churned_df_encoded[churned_df_encoded['Attrition'] == 0].sample(n=val_undersample, random_state=42).index
df_undersample = churned_df_encoded.drop(rows_to_drop)

In [23]:
df_undersample = df_undersample.reindex(sorted(df_undersample.columns), axis=1)

In [24]:
churned_df = df_undersample.copy()[['AgeGeneration', 'Attrition', 'BusinessTravel', 'DailyRate',
       'Department', 'DistanceFromHome', 'DistanceFromHomeCategory',
       'Education', 'EducationField', 'EnvironmentSatisfaction',
        'Gender', 'HourlyRate', 'JobInvolvement',
       'JobLevel', 'JobRole', 'JobSatisfaction', 'MaritalStatus', 'NumCompaniesWorked',
    'OverTime', 'Overtime_Preference', 'PercentSalaryHike',
       'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager']]

In [25]:
val_count = int(churned_df.shape[0] * .10)

In [26]:
df_encoded_val = churned_df.sample(n=val_count, random_state=42)
df_encoded_train = churned_df.drop(df_encoded_val.index)

In [27]:
models = [LogisticRegression(random_state=42), RandomForestClassifier(random_state=42), GaussianNB(), DecisionTreeClassifier(random_state=42), GradientBoostingClassifier(random_state=42), AdaBoostClassifier(random_state=42)]

In [28]:
 X_train, X_test, y_train, y_test = train_test_split(df_encoded_train[features], df_encoded_train[target], test_size=0.2, random_state=42)

for index in range(len(models)):
    print('------------------------------------')
    print(type(models[index]))
    models[index].fit(X_train, y_train)
    
    y_pred = models[index].predict(X_test)
    y_pred_proba =  models[index].predict_proba(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    print()
    print(f'Accuracy: {accuracy:.2f}')
    print(f'Precision: {precision:.2f}')
    print(f'Recall: {recall:.2f}')
    print(f'F1 Score: {f1:.2f}')
    
    cm = confusion_matrix(y_test, y_pred)
    print()
    
    print(cm)
    '''
    plt.clf()

    sns.set(font_scale=.8)
    plt.figure(figsize=(6,6))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["Active", "Churned"], yticklabels=["Active", "Churned"])
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix of Training Data')
    plt.show()'''
    
    if  index != 0 | index != 2 :
        importance_scores =  models[index].feature_importances_
    
        # Create a dataframe to display the feature importance scores
        importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': importance_scores})
        importance_df = importance_df.sort_values('Importance', ascending=False)
        print()
        print(importance_df)

------------------------------------
<class 'sklearn.linear_model._logistic.LogisticRegression'>

Accuracy: 0.69
Precision: 0.67
Recall: 0.62
F1 Score: 0.64

[[35 12]
 [15 24]]
------------------------------------
<class 'sklearn.ensemble._forest.RandomForestClassifier'>


lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



Accuracy: 0.74
Precision: 0.71
Recall: 0.74
F1 Score: 0.72

[[35 12]
 [10 29]]
------------------------------------
<class 'sklearn.naive_bayes.GaussianNB'>

Accuracy: 0.70
Precision: 0.63
Recall: 0.79
F1 Score: 0.70

[[29 18]
 [ 8 31]]
------------------------------------
<class 'sklearn.tree._classes.DecisionTreeClassifier'>

Accuracy: 0.67
Precision: 0.63
Recall: 0.67
F1 Score: 0.65

[[32 15]
 [13 26]]
------------------------------------
<class 'sklearn.ensemble._gb.GradientBoostingClassifier'>

Accuracy: 0.67
Precision: 0.64
Recall: 0.64
F1 Score: 0.64

[[33 14]
 [14 25]]
------------------------------------
<class 'sklearn.ensemble._weight_boosting.AdaBoostClassifier'>

Accuracy: 0.74
Precision: 0.73
Recall: 0.69
F1 Score: 0.71

[[37 10]
 [12 27]]


The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


In [29]:
estimators_models = [('lr',LogisticRegression(random_state=42)), ('rfc', RandomForestClassifier(random_state=42)), 
              ('gnb', GaussianNB()), ('dtc', DecisionTreeClassifier(random_state=42)), ('gbc', GradientBoostingClassifier(random_state=42)),
                     ('abc', AdaBoostClassifier(random_state=42))]

In [30]:
model = VotingClassifier(
        estimators=estimators_models,
        voting='hard'
    )

print(type(model))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print()
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

cm = confusion_matrix(y_test, y_pred)
print()

print(cm)

<class 'sklearn.ensemble._voting.VotingClassifier'>


lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression



Accuracy: 0.74
Precision: 0.73
Recall: 0.69
F1 Score: 0.71

[[37 10]
 [12 27]]


The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


In [31]:
model = VotingClassifier(
        estimators=estimators_models,
        voting='soft'
    )

print(type(model))
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba =  model.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print()
print(f'Accuracy: {accuracy:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'F1 Score: {f1:.2f}')

cm = confusion_matrix(y_test, y_pred)
print()

print(cm)


lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression


<class 'sklearn.ensemble._voting.VotingClassifier'>

Accuracy: 0.74
Precision: 0.70
Recall: 0.77
F1 Score: 0.73

[[34 13]
 [ 9 30]]


The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.


In [34]:
import shap

In [47]:
# Initialize KernelExplainer with a sample of training data
explainer = shap.KernelExplainer(model.predict, shap.sample(X_train, 170), keep_index=True)

# Compute SHAP values for the test data
shap_values = explainer.shap_values(X_test)

Using 170 background data samples could cause slower run times. Consider using shap.sample(data, K) or shap.kmeans(data, K) to summarize the background as K samples.


  0%|          | 0/86 [00:00<?, ?it/s]

AxisError: axis 1 is out of bounds for array of dimension 1

In [48]:
# Calculate absolute SHAP values and convert to percentages
shap_values_abs = np.abs(shap_values)
shap_values_percent = (shap_values_abs / shap_values_abs.sum(axis=1, keepdims=True)) * 100

# Convert to DataFrame and name columns based on feature names
shap_values_percent_df = pd.DataFrame(shap_values_percent, columns=features)

# Compute average feature importance
feature_importance_avg = shap_values_percent_df.mean().sort_values(ascending=False).reset_index()
feature_importance_avg.columns = ['Feature', 'Importance']
feature_importance_avg['Importance'] = feature_importance_avg['Importance'].apply(lambda x: f"{x:.2f}%")

# Rename SHAP DataFrame columns to include feature importance in %
shap_values_percent_df.columns = [f"Feature Importance of {col} in %" for col in shap_values_percent_df.columns]

# Display the average feature importance
print(feature_importance_avg)

# Display the SHAP values DataFrame
print(shap_values_percent_df.head())

                     Feature Importance
0        Overtime_Preference     17.09%
1          TotalWorkingYears      9.91%
2         YearsInCurrentRole      5.98%
3      TrainingTimesLastYear      5.53%
4                   JobLevel      5.37%
5            JobSatisfaction      4.66%
6                    JobRole      4.43%
7            WorkLifeBalance      4.24%
8           StockOptionLevel      3.87%
9             EducationField      3.65%
10  RelationshipSatisfaction      3.26%
11                 DailyRate      3.20%
12        NumCompaniesWorked      2.88%
13            JobInvolvement      2.77%
14                Department      2.71%
15            YearsAtCompany      2.60%
16                    Gender      2.47%
17  DistanceFromHomeCategory      2.44%
18             MaritalStatus      2.41%
19   EnvironmentSatisfaction      2.28%
20      YearsWithCurrManager      2.27%
21         PercentSalaryHike      1.98%
22   YearsSinceLastPromotion      1.67%
23            BusinessTravel      0.82%


In [49]:
feature_importance = np.abs(shap_values).mean(axis=0)
df_feature_importance = pd.DataFrame(feature_importance, index=features, columns=['feature_importance'])
df_feature_importance.sort_values(by='feature_importance', ascending=False, inplace=True)

In [57]:
df_feature_importance.loc['Overtime_Preference'].values[0]

0.16723039399241338

# Export

In [32]:
filename_model = "../models/model_{}.sav".format(datetime.now().strftime('%Y_%m_%d'))
with open(filename_model, 'wb') as f:
    pickle.dump(model, f)

In [33]:
filename_encoder = "../models/encoder_{}.sav".format(datetime.now().strftime('%Y_%m_%d'))
with open(filename_encoder, 'wb') as f:
    pickle.dump(encoders, f)

In [52]:
filename_explainer = "../models/explainer_{}.sav".format(datetime.now().strftime('%Y_%m_%d'))
with open(filename_explainer, 'wb') as f:
    pickle.dump(explainer, f)